# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RawanMohamed16/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just finds the repo root
from wherever this notebook happens to be opened. Run this first.


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/RawanMohamed16/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # walk up from wherever this kernel started until we find the repo root
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /home/claude/test_run/flyrank-ml-internship
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring.**

I'm picking this lane because it turns a huge, unreviewable pile of pages into a short, ranked
list a human can actually act on — and because the starter pipeline in this repo already proves
the workflow works end to end (see the numbers below). It also gives me the clearest possible
version of the assignment's four required pieces: a real decision (what to review first), a real
action (refresh / expand / protect / prune / monitor), a real cost of getting it wrong (wasted
reviewer time, or a real opportunity left un-reviewed), and a way to check my work honestly
(precision@K against a held-out set of clients). Compared to signal analysis or clustering, this
lane has the tightest link between "what the data shows" and "what a person does next."


In [2]:
# Quick sanity check: the starter pipeline (scripts/01-05) already ran on this same CSV
# and committed its results to outputs/model_report.md. I'm not re-deriving these here —
# just confirming the file exists and pulling the headline comparison so I can cite it honestly.

with open("outputs/model_report.md") as f:
    report = f.read()

# Pull just the model comparison table out of the report
start = report.index("## Model Comparison")
end = report.index("## Final Queue")
print(report[start:end].strip())


## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Question:** Given a page's observed search and engagement signals, which pages should a content
reviewer look at *first* — and with what action (refresh, expand, protect, prune, or monitor)?

- **Decision improved:** which of thousands of existing pages get a human reviewer's limited
  attention this week, instead of reviewers scanning pages in an arbitrary or alphabetical order.
- **Who acts on it:** a content/SEO reviewer (or account manager) with a fixed weekly review
  capacity — they can realistically check maybe 20-50 pages, not 30,000.
- **Cost of a wrong call:**
  - *False positive* (flagged as urgent, isn't really): wastes a reviewer's limited time on a page
    that didn't need attention, at the expense of a page that did.
  - *False negative* (a real opportunity never surfaces): a genuinely declining or under-performing
    page with real demand keeps losing visibility, unnoticed, until it's a bigger problem.
  - Because the cost is about *ranking* and *attention*, not a one-shot yes/no call, precision at
    the top of the list (precision@K) matters more than overall accuracy.


In [3]:
# No computation needed for this section — it's framing, not data.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*


In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
n = len(df)

# 1) How big is the backlog? Share of pages currently trending down.
declining = (df["trend_direction"] == "down").sum()

# 2) Of those, how many have REAL demand behind them (not just noise)?
#    impressions_90d >= 100 is the same "declining_with_demand" rule the starter baseline uses.
declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()

# 3) A second, distinct opportunity type: decent visibility but weak click-through.
#    impressions_90d >= 500, position 1-20, ctr < 0.5% -> the starter's "low_ctr_visible_page" rule.
low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).sum()

print(f"Total pages in starter dataset: {n:,}")
print(f"1) Declining pages: {declining:,} ({declining/n:.1%} of all pages)")
print(f"2) ...of which have real demand (>=100 impressions/90d): {declining_with_demand:,} ({declining_with_demand/n:.1%} of all pages)")
print(f"3) Visible-but-weak-CTR pages (separate opportunity type): {low_ctr_visible:,} ({low_ctr_visible/n:.1%} of all pages)")


Total pages in starter dataset: 30,000
1) Declining pages: 16,262 (54.2% of all pages)
2) ...of which have real demand (>=100 impressions/90d): 13,152 (43.8% of all pages)
3) Visible-but-weak-CTR pages (separate opportunity type): 9,759 (32.5% of all pages)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- Observed patterns: "these pages show declining impressions/clicks/position over the observed
  window" — a measurement, not a guess.
- Directional, decision-support recommendations: "this page is a good candidate to review first,
  based on these signals" — a ranking to help a human spend limited time well.
- A comparison against a transparent baseline rule, with an honest metric (precision@K) that
  matches how the list will actually be used.

**What I can never claim:**
- That a refresh *caused* a recovery — that needs an experiment (e.g. before/after with a control
  group), which this dataset alone cannot give me.
- That I've reverse-engineered or predicted Google's ranking algorithm.
- That the model's score is "the truth" — it's a ranked hypothesis for a human to check, not a
  verdict.
- Anything about a specific client, domain, URL, or query — the data is pseudonymized and I'll
  only ever talk about IDs, aggregates, and safe examples.


In [5]:
# No computation needed for this section either — it's a statement of scope, not a data check.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.